# 03 — BERT Fine-tuning for Sentiment Classification

This notebook fine-tunes `bert-base-uncased` on the sentiment dataset
and compares it against the baseline.

**Requirements:** GPU strongly recommended (≥ 6 GB VRAM).  
The code auto-detects CPU/GPU and falls back gracefully.

Steps:
1. Load data
2. Train BERT
3. Evaluate on test set
4. Compare with baseline
5. Error analysis

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd

from config import set_seed
from src.data_loader import load_data, split_data
from src.bert_model import train_bert, predict_bert, load_bert_model
from src.evaluate import evaluate_model, plot_confusion_matrix, compare_models, plot_roc_curve

set_seed()
print(f'Torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Load Data

In [ ]:
df = load_data()
train_df, val_df, test_df = split_data(df)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

# Use a smaller subset for quick notebook runs
QUICK_RUN = True
if QUICK_RUN:
    train_df = train_df.sample(min(1000, len(train_df)), random_state=42).reset_index(drop=True)
    val_df = val_df.sample(min(200, len(val_df)), random_state=42).reset_index(drop=True)
    print(f'Quick run — using {len(train_df)} train / {len(val_df)} val samples.')

## 2. Fine-tune BERT

This cell downloads `bert-base-uncased` (~440 MB) on the first run.

In [ ]:
model, tokenizer = train_bert(
    train_df, val_df,
    config={'epochs': 2, 'batch_size': 16},
)
print('Training complete.')

## 3. Test Set Evaluation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
y_test = test_df['label'].tolist()

bert_preds, bert_confs = predict_bert(
    model, tokenizer,
    test_df['clean_text'].tolist(),
    device=device,
)

bert_results = evaluate_model(y_test, bert_preds, 'BERT Fine-tuned', y_scores=bert_confs)
pd.Series(bert_results, name='Score').to_frame()

## 4. Confusion Matrix

In [ ]:
from IPython.display import Image
cm_path = plot_confusion_matrix(y_test, bert_preds, 'BERT Fine-tuned', labels=['Negative', 'Positive'])
Image(str(cm_path))

## 5. ROC Curve

In [ ]:
roc_path = plot_roc_curve(y_test, bert_confs, 'BERT Fine-tuned')
Image(str(roc_path))

## 6. Model Comparison

Load baseline results and compare side-by-side.

In [ ]:
import joblib
from config import BASELINE_MODEL_PATH
from src.baseline_model import predict as predict_baseline

try:
    baseline_pipeline = joblib.load(BASELINE_MODEL_PATH)
    bl_preds, bl_probs = predict_baseline(baseline_pipeline, test_df['clean_text'].tolist())
    bl_scores = bl_probs[:, 1] if bl_probs.shape[1] == 2 else bl_probs.max(axis=1)
    baseline_results = evaluate_model(y_test, bl_preds, 'Baseline', y_scores=bl_scores)
    
    from IPython.display import Image as IPImage
    comparison_path = compare_models(baseline_results, bert_results)
    print(comparison_path)
    from config import FIGURES_DIR
    IPImage(str(FIGURES_DIR / 'model_comparison.png'))
except FileNotFoundError:
    print('Baseline model not found — run notebook 02 first.')

## 7. Error Analysis — Misclassified Examples

In [ ]:
import numpy as np

errors_mask = bert_preds != np.array(y_test)
error_df = test_df[errors_mask].copy()
error_df['predicted'] = bert_preds[errors_mask]
error_df['confidence'] = bert_confs[errors_mask]
error_df = error_df.sort_values('confidence', ascending=False)

label_map = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
error_df['true_label'] = error_df['label'].map(label_map)
error_df['pred_label'] = error_df['predicted'].map(label_map)

print(f'Total misclassifications: {len(error_df)} / {len(test_df)} ({len(error_df)/len(test_df)*100:.1f}%)')
error_df[['clean_text', 'true_label', 'pred_label', 'confidence']].head(10)